In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pendulum as pdl
import datetime as dt

In [3]:
# Create a dataframe which defines the 
loadcase_timesheet = pd.DataFrame(columns=["LC","lc-title","timestamp"])
sofistik_results = pd.read_csv("openlab-sensors-export.csv")
# the sofistic produces CSV files with:
#  LC, Node, u-X, u-Y, u-Z, r-X, r-Y, r-Z, ...
# loaded_data = pd.read_csv("test-openlab-export.csv",)
loaded_data = pd.read_csv("openlab-sensors-export.csv",)
# del the first 2/3 of records
# len(loaded_data)//3*2
# loaded_data = loaded_data.iloc[len(loaded_data)//3*2:, :]
# len(loaded_data)


In [ ]:
def find_min_lc_num(dftemp: pd.DataFrame = sofistik_results.copy()) -> int:
    """
    Find the minimum loadcase number in the loadcase_timesheet.
    
    Args:
        dftemp (pd.DataFrame): The dataframe from which to find the minimum loadcase number.
        
    Returns:
        int: The minimum loadcase number in the loadcase_timesheet. If the dataframe is empty, returns 0.
    """
    dftemp = dftemp.query("`LC-title ` == 'Creep until start of load test'")
    if not dftemp.empty:
        return dftemp["LC "].min()
    else:
        raise ValueError("The loadcase_timesheet is empty or does not contain the specified loadcase title.")

def add_relevant_loadcases(loadcase_timesheet, lc_title: str, start_date: dt.datetime, simulation_duration: int, simulation_steps: int, start_lc_num: int = None) -> pd.DataFrame:
    """
    Add relevant loadcases to the loadcase_timesheet
    
    Args:
        loadcase_timesheet (pd.DataFrame): The loadcase timesheet to which the loadcases will be added.
        lc_title (str): The title of the loadcase to be added
        start_date (dt.datetime): The start date of the simulation.
        simulation_duration (pdl.Duration): The duration of the simulation in days/second/...
        simulation_steps (int): The number of steps in the simulation.
    Returns:
        pd.DataFrame: The updated loadcase_timesheet with the added loadcases.
    """
    start_lc_num = find_min_lc_num() if start_lc_num is None else start_lc_num
    for i in range(1, simulation_steps + 1):
        loadcase_timesheet.loc[simulation_steps + i] = {
            "LC": start_lc_num + i,
            "lc-title": lc_title,
            "timestamp": (
                start_date + (i-1) * simulation_duration / simulation_steps
                ).to_datetime_string(),
            # pdl.Duration(
            #     # days=simulation_duration + i * simulation_duration / simulation_steps
            #     days=simulation_duration + i * simulation_duration / simulation_steps
            # ),
        }
    return loadcase_timesheet

# start_date = pd.to_datetime("2023-11-01 00:00:00")
# # create the 10 loadsteps for the creep load case
# simulation_duration = 365 #days
# simulation_steps = 30
# for i in range(1,simulation_steps+1):
#     loadcase_timesheet.loc[i - 1] = {
#         "LC": 40074 + i,
#         "lc-title": "Creep until start of load test",
#         "timestamp": start_date + i * pdl.Duration(
#             days=simulation_duration / simulation_steps
#         ),
#     }

In [19]:
# DEFINE THE CONSTRUCTION HISTORY - i.e. loadcases which you want to extract

HISTORY = [
    {
        "lc_title": "Creep until start of load test",
        "start_date": pdl.parse("2023-11-27 00:00:00"), # Assembly of all PE in OpenLAB timeline.csv
        "simulation_duration": pdl.duration(days=365),
        "simulation_steps": 30,
    },
    {
        "lc_title": f"Heavier vehicle crossing",
        "start_date": pdl.parse("2024-10-23 10:52:38"),
        "simulation_duration": pdl.duration(seconds=45),
        "simulation_steps": 60,
        "start_lc_num": 301,
    } #TO DO: REFLECT THE LOADCASE TO OBTAIN THE COMPLETE forth-back CROSSING
]

# HISTORY.append([{
#     "lc_title": f"Influence line xm =   {1.1+i*0.5:.2f} m",
#     "start_date": pdl.parse("2024-10-23 10:52:38"),
#     "simulation_duration": pdl.duration(seconds=90),
#     "simulation_steps": 60,
# } for i in range(60)])

In [23]:
for record in HISTORY:
    loadcase_timesheet = add_relevant_loadcases(
        loadcase_timesheet,
        lc_title=record["lc_title"],
        start_date=record["start_date"],
        simulation_duration=record["simulation_duration"],
        simulation_steps=record["simulation_steps"],
        start_lc_num=record.get("start_lc_num", None)
    )

# loadcase_timesheet = add_relevant_loadcases(
#     loadcase_timesheet,
#     lc_title="Creep until start of load test",
#     # start_lc_num=40074,
#     start_date=pd.to_datetime("2023-11-01 00:00:00"),
#     simulation_duration=365, #days
#     simulation_steps=30,
# )
print(loadcase_timesheet.timestamp.dtype)
print(loadcase_timesheet.shape)
loadcase_timesheet.head(5)
loadcase_timesheet.tail(5)

TypeError: DateTime.to_datetime_string() takes 1 positional argument but 2 were given

In [7]:
# save the loadcase_timesheet to a CSV file
loadcase_timesheet.to_csv("loadcase_timesheet.csv", index=False)

In [8]:
sensors = pd.read_csv("Sensor_Names_autoextracted_from_model.csv",)
# loaded data format correction|
p = loaded_data.columns
loaded_data[p[3:]] = loaded_data[p[3:]].apply(lambda x: x.str.replace(',', '.'))
loaded_data[p[3:]] = loaded_data[p[3:]].apply(lambda x: pd.to_numeric(x, errors='coerce'))


def from_mrad_to_mm_per_m(mrad):
    """Convert mrad to mm/m"""
    # mrad.str.replace(",", ".", inplace=True)
    values = pd.to_numeric(mrad, errors="raise")
    return np.tan(values * 1e-3) * 1000

for _, sensor in sensors.iterrows():
    # get senor ref name
    refnum = sensor["sofistik_ref_number"]
    if refnum is None:
        continue
    # extract the relevant predictions
    dftemp = loaded_data["NODE "] == refnum
    dftemp = loaded_data[dftemp]
    if dftemp.empty:
        continue
    else:
        dftemp = dftemp.query("`LC-title ` == 'Creep until start of load test'")
        dftemp = dftemp.copy()
        QoI_dict = {
            "Tilt-X": from_mrad_to_mm_per_m(dftemp["phi-Y [mrad]"].values),
            "Tilt-Y": from_mrad_to_mm_per_m(dftemp["phi-X [mrad]"].values),
            "Displacement-X": dftemp["u-X [mm]"].values,
            "Displacement-Y": dftemp["u-Y [mm]"].values,
            "Displacement-Z": dftemp["u-Z [mm]"].values,
        }
        SensorNameDict = {
            "NGS": "TIL",
            "FT": "PE",
            "BU": "CB",
        }
        sensor_name = sensor["Name"]
        for key, value in SensorNameDict.items():
            sensor_name = sensor_name.replace(key, value)
        # create a new column with the sensor name
        try:
            qoi = QoI_dict[sensor["QoI"]]
        except KeyError:
            print(f"QoI {sensor['QoI']} not found for sensor {sensor['Name']}")
            continue
        loadcase_timesheet[sensor_name] = qoi
        
loadcase_timesheet.head(11)


,LC,lc-title,timestamp,G_TILX_PE11_CB1100_0,G_TILY_PE11_CB1100_0,G_TILX_PE12_CB1100_0,G_TILY_PE12_CB1100_0,G_TILX_PE13_CB1100_0,G_TILY_PE13_CB1100_0,G_TILX_PE21_CB0400_0,G_TILY_PE21_CB0400_0,G_TILX_PE22_CB0400_0,G_TILY_PE22_CB0400_0,G_TILX_PE23_CB0400_0,G_TILY_PE23_CB0400_0
31,40076,Creep until start of load test,2023-11-27 00:00:00+00:00,0.480,0.061,0.477,0.009,0.468,-0.044,-0.980,0.021,-0.982,-0.024,-0.978,-0.068
32,40077,Creep until start of load test,2023-12-09 04:00:00+00:00,0.493,0.061,0.489,0.009,0.481,-0.043,-1.007,0.020,-1.009,-0.025,-1.004,-0.068
33,40078,Creep until start of load test,2023-12-21 08:00:00+00:00,0.503,0.062,0.499,0.009,0.490,-0.043,-1.028,0.020,-1.030,-0.026,-1.025,-0.069
34,40079,Creep until start of load test,2024-01-02 12:00:00+00:00,0.512,0.062,0.507,0.009,0.498,-0.043,-1.047,0.019,-1.050,-0.027,-1.045,-0.070
35,40080,Creep until start of load test,2024-01-14 16:00:00+00:00,0.519,0.062,0.513,0.009,0.504,-0.043,-1.064,0.018,-1.067,-0.028,-1.062,-0.071
36,40081,Creep until start of load test,2024-01-26 20:00:00+00:00,0.525,0.062,0.519,0.010,0.509,-0.043,-1.079,0.017,-1.081,-0.029,-1.077,-0.072
37,40082,Creep until start of load test,2024-02-08 00:00:00+00:00,0.531,0.062,0.524,0.010,0.514,-0.043,-1.094,0.016,-1.096,-0.030,-1.091,-0.073
38,40083,Creep until start of load test,2024-02-20 04:00:00+00:00,0.536,0.062,0.528,0.010,0.519,-0.043,-1.107,0.015,-1.109,-0.031,-1.104,-0.073
39,40084,Creep until start of load test,2024-03-03 08:00:00+00:00,0.540,0.062,0.532,0.010,0.522,-0.043,-1.118,0.014,-1.121,-0.031,-1.116,-0.074
40,40085,Creep until start of load test,2024-03-15 12:00:00+00:00,0.544,0.062,0.536,0.010,0.526,-0.043,-1.129,0.014,-1.131,-0.032,-1.126,-0.075


In [9]:
# prform a join between the two dataframes to map the LC to timestamps 
# loaded_data.join(loadcase_timesheet.set_index("LC"), on=0, how="left", rsuffix="_timestamp")
        
loaded_data[loaded_data["LC-title "]=="Creep until start of load test" ].head()
cpdf = loaded_data[loaded_data["LC-title "]=="Creep until start of load test" ].copy()

cpdf["LC "] = cpdf["LC "].astype(str).str.strip()
loadcase_timesheet["LC"] = loadcase_timesheet["LC"].astype(str).str.strip()

timestamp_map = loadcase_timesheet.set_index("LC")["timestamp"]

print(timestamp_map.dtype)

# cpdf["timestamp"] = cpdf["LC "].map(timestamp_map)
cpdf["timestamp"] = cpdf["LC "].map(loadcase_timesheet.set_index("LC")["timestamp"])
cpdf.head()

datetime64[us, UTC]


,LC,LC-title,NODE,u-X [mm],u-Y [mm],u-Z [mm],phi-X [mrad],phi-Y [mrad],phi-Z [mrad],timestamp
264,40075,Creep until start of load test,1140,-0.880,0.006,1.891,0.061,0.480,0.000,NaT
265,40075,Creep until start of load test,1250,-1.803,0.003,4.248,0.021,-0.980,-0.001,NaT
266,40075,Creep until start of load test,2140,-0.803,0.003,1.951,0.009,0.477,0.000,NaT
267,40075,Creep until start of load test,2250,-1.651,-0.001,4.251,-0.024,-0.982,0.000,NaT
268,40075,Creep until start of load test,3140,-0.796,-0.001,1.918,-0.044,0.468,-0.001,NaT


In [10]:
print(cpdf.columns)
cpdf["timestamp"].head()


Index(['LC ', 'LC-title ', 'NODE ', 'u-X [mm]', 'u-Y [mm]', 'u-Z [mm]',
       'phi-X [mrad]', 'phi-Y [mrad]', 'phi-Z [mrad]', 'timestamp'],
      dtype='str')


264   NaT
265   NaT
266   NaT
267   NaT
268   NaT
Name: timestamp, dtype: datetime64[us, UTC]

In [11]:
# cpdf.plot(x="LC ", y=['u-X [mm]', 'u-Y [mm]', 'u-Z [mm]'],
#           title="Displacement over time for Creep Load Case",
#           xlabel="Load Case", ylabel="Displacement [mm]", marker='o', linestyle='--', figsize=(10, 4))

# rotation_columns = ["phi-X [mrad]", "phi-Y [mrad]", "phi-Z [mrad]"]
# # .assign(y_scaled=lambda d: d[rotation_columns[2]] * 10).

# cpdf.plot(x="LC ", y=rotation_columns,
#           title="Displacement over time for Creep Load Case",
#           xlabel="Load Case", ylabel="Rotation [mrad*10]", marker='o', linestyle='--', figsize=(10, 4))

In [12]:
cpdf.columns
loadcase_timesheet.head(5)


,LC,lc-title,timestamp,G_TILX_PE11_CB1100_0,G_TILY_PE11_CB1100_0,G_TILX_PE12_CB1100_0,G_TILY_PE12_CB1100_0,G_TILX_PE13_CB1100_0,G_TILY_PE13_CB1100_0,G_TILX_PE21_CB0400_0,G_TILY_PE21_CB0400_0,G_TILX_PE22_CB0400_0,G_TILY_PE22_CB0400_0,G_TILX_PE23_CB0400_0,G_TILY_PE23_CB0400_0
31,40076,Creep until start of load test,2023-11-27 00:00:00+00:00,0.480,0.061,0.477,0.009,0.468,-0.044,-0.980,0.021,-0.982,-0.024,-0.978,-0.068
32,40077,Creep until start of load test,2023-12-09 04:00:00+00:00,0.493,0.061,0.489,0.009,0.481,-0.043,-1.007,0.020,-1.009,-0.025,-1.004,-0.068
33,40078,Creep until start of load test,2023-12-21 08:00:00+00:00,0.503,0.062,0.499,0.009,0.490,-0.043,-1.028,0.020,-1.030,-0.026,-1.025,-0.069
34,40079,Creep until start of load test,2024-01-02 12:00:00+00:00,0.512,0.062,0.507,0.009,0.498,-0.043,-1.047,0.019,-1.050,-0.027,-1.045,-0.070
35,40080,Creep until start of load test,2024-01-14 16:00:00+00:00,0.519,0.062,0.513,0.009,0.504,-0.043,-1.064,0.018,-1.067,-0.028,-1.062,-0.071


In [13]:
# save the csv dataframe
import os
os.remove("tiltmeters_prediction.csv")
loadcase_timesheet.to_csv("tiltmeters_prediction.csv", index=False)

In [14]:
loadcase_timesheet.columns

Index(['LC', 'lc-title', 'timestamp', 'G_TILX_PE11_CB1100_0',
       'G_TILY_PE11_CB1100_0', 'G_TILX_PE12_CB1100_0', 'G_TILY_PE12_CB1100_0',
       'G_TILX_PE13_CB1100_0', 'G_TILY_PE13_CB1100_0', 'G_TILX_PE21_CB0400_0',
       'G_TILY_PE21_CB0400_0', 'G_TILX_PE22_CB0400_0', 'G_TILY_PE22_CB0400_0',
       'G_TILX_PE23_CB0400_0', 'G_TILY_PE23_CB0400_0'],
      dtype='str')

## POSTPROCESS THE VEHICLE CROSSING TEST
...


In [15]:
# 